In [1]:
print("ok")

ok


In [2]:
%pwd

'd:\\CODE\\PROJECTS\\MEDICHAT\\research'

In [3]:
import os
os.chdir("../")

In [4]:
%pwd

'd:\\CODE\\PROJECTS\\MEDICHAT'

In [5]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI


In [6]:
#Extract Data From the PDF File
def load_pdf_file(data):
        loader= DirectoryLoader(data,
                                glob="*.pdf",
                                loader_cls=PyPDFLoader)
        documents=loader.load()
        return documents

In [7]:
extracted_data=load_pdf_file(data='KnowledgeBase/')

In [8]:
# extracted_data

In [9]:
#Split the Data into Text Chunks
def text_split(extracted_data):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks=text_splitter.split_documents(extracted_data)
    return text_chunks

In [10]:
# chunk = text_chunks[0]
# vector = embeddings.embed_query(chunk.page_content)
# print(len(vector))


In [11]:
# for i, chunk in enumerate(text_chunks):
#     print(f"Chunk {i} length: {len(chunk.page_content)}")


In [12]:
text_chunks=text_split(extracted_data)
print("Length of Text Chunks", len(text_chunks))


Length of Text Chunks 47490


In [13]:
from langchain.embeddings import HuggingFaceEmbeddings

In [14]:
#Download the Embeddings from Hugging Face
def download_hugging_face_embeddings():
    embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    return embeddings

In [15]:
embeddings = download_hugging_face_embeddings()

C:\Users\MANIS\AppData\Local\Temp\ipykernel_23556\2661704553.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
c:\Users\MANIS\anaconda3\envs\medichat\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
query_result = embeddings.embed_query("Hello world")
print("Length", len(query_result))

Length 384


In [17]:
#query_result

In [18]:
from dotenv import load_dotenv
load_dotenv()

True

In [19]:
import os
PINECONE_API_KEY=os.environ.get('PINECONE_API_KEY')
GEMINI_API_KEY=os.environ.get('GEMINI_API_KEY')

In [21]:
import os
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec
pc = Pinecone(api_key=PINECONE_API_KEY)
index_name = "medichat"

pc.create_index(
    name=index_name,
    dimension=384,
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    ) 
)

PineconeApiException: (409)
Reason: Conflict
HTTP response headers: HTTPHeaderDict({'content-type': 'text/plain; charset=utf-8', 'access-control-allow-origin': '*', 'vary': 'origin,access-control-request-method,access-control-request-headers', 'access-control-expose-headers': '*', 'x-pinecone-api-version': '2025-01', 'x-cloud-trace-context': 'c427210c23994b1dd5675a2b7a003924', 'date': 'Thu, 24 Jul 2025 05:12:11 GMT', 'server': 'Google Frontend', 'Content-Length': '85', 'Via': '1.1 google', 'Alt-Svc': 'h3=":443"; ma=2592000,h3-29=":443"; ma=2592000'})
HTTP response body: {"error":{"code":"ALREADY_EXISTS","message":"Resource  already exists"},"status":409}


In [22]:
import os
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

In [23]:
# Embed each chunk and upsert the embeddings into your Pinecone index.
from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    index_name=index_name,
    embedding=embeddings,
)


In [24]:
# Load Existing index
from langchain_pinecone import PineconeVectorStore
#Embed each chunk and upsert the embeddings into your Pinecone index.
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

In [25]:
docsearch

In [26]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [27]:
retrieved_docs = retriever.invoke("What is Acne?")

In [28]:
retrieved_docs

[Document(id='407cbf70-a6fa-4a83-81be-0e4bfa12c0ca', metadata={'creationdate': '', 'creator': 'PyPDF', 'moddate': '2025-04-29T14:22:20+00:00', 'page': 650.0, 'page_label': '651', 'producer': 'iLovePDF', 'source': 'KnowledgeBase\\Harrison.pdf', 'total_pages': 6390.0}, page_content="ACNE (Table 57-7)\nAcne vulgaris and acne rosacea are the two major forms of acne (Chap. 56). Estrogens \ndecrease sebaceous gland activity, whereas androgens enhance sebum production. \nTherefore, acne vulgaris in an adult, especially if it is of recent onset, may be a \nreflection of increased levels of circulating androgens. Dysfunction of the ovary or \nadrenal gland, e.g., polycystic ovary disease or Cushing's syndrome, can lead to the"),
 Document(id='adf49f80-a0af-4c4c-a3a8-cdaeeafdc380', metadata={'creationdate': '', 'creator': 'PyPDF', 'moddate': '2025-04-29T14:22:20+00:00', 'page': 650.0, 'page_label': '651', 'producer': 'iLovePDF', 'source': 'KnowledgeBase\\Harrison.pdf', 'total_pages': 6390.0}, pa

In [29]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.4, max_output_tokens=500, google_api_key=GEMINI_API_KEY)

In [30]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
system_prompt = (
    "You are a medical assistant. Your task is to provide accurate and concise answers to medical questions based on the provided context.\n"
    "You should not provide any personal opinions or make assumptions beyond the information given.\n"
    "the data source you have is coming from various books like Harrison's Principles of Internal Medicine, Gale encyclopedia of medicine, and Merck Manual.\n"
    "You are MediBot, an intelligent and factual AI assistant designed to offer concise, medically relevant answers.\n"
    "If the user greets (e.g., 'hi', 'hello'), reply briefly and politely.\n"
    "If the user apologizes (e.g., 'sorry'), acknowledge politely and offer help.\n"
    "If the user asks about the source of data or context, describe what kind of documents or data have been used (e.g., patient FAQs, medical research, or health guidelines).\n"
    "Otherwise, read the CONTEXT carefully and answer the QUESTION in no more than four sentences.\n"
    "You may include brief explanations or reasoning steps if helpful, and refer to the source type where possible.\n"
    "Be strictly based on the CONTEXT—do not make up information. If the CONTEXT does not contain the answer, say 'I don’t know.'\n\n"
    "CONTEXT:\n{context}\n\n"
)
prompt = ChatPromptTemplate.from_messages(
                         [
                             ("system", system_prompt),
                             ("human", "{input}"),     
                         ] 
)

In [31]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [34]:
response = rag_chain.invoke({"input": "What is Acne?"})
print(response["answer"])

Acne vulgaris and acne rosacea are the two major forms of acne (per Table 57-7). Small cysts formed in hair follicles are a clinical hallmark. Acne is often accompanied by inflammatory lesions of papules, pustules, or nodules and may scar in severe cases. Careful cleaning and removal of oils, oral tetracycline or erythromycin, topical antibacterials (e.g., benzoyl peroxide), and topical retinoic acid can treat it.


In [38]:
response = rag_chain.invoke({"input": "What is the cure for fever, and medicine."})
print(response["answer"])

Treating fever and its symptoms does no harm and does not slow the resolution of common viral and bacterial infections (from the book Harrison's Principles of Internal Medicine). Reducing fever with antipyretics also reduces systemic symptoms of headache, myalgias, and arthralgias. Drug treatment is the foremost noninfectious cause of fever.  Particular agents associated with drug fever include phenytoin, H2blockers, procainamide, and antibiotics, most notably.


In [40]:
response = rag_chain.invoke({"input": "What is agentic ai?"})
print(response["answer"])

I'm sorry, but this document does not contain a definition of agentic AI. However, it does mention that artificial intelligence systems attempt to aid in the decision-making process or provide algorithmic guidance. (Various books)
